# Solution 02 — the model shoot-out

Same prompt through three model strings, `result.usage` collected into a table.
Swap in whatever models your key actually supports.

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

In [ ]:
from pydantic_ai import Agent

prompt = 'Explain what a "token" is, in exactly one sentence.'
candidates = {
    "primary (MODEL)": MODEL,
    "gpt-5.6-sol     ": "openai:gpt-5.6-sol",
    "gpt-5.5         ": "openai:gpt-5.5",
}

rows = []
for label, model_string in candidates.items():
    candidate = Agent(model_string, instructions="Answer with exactly one sentence.")
    r = await candidate.run(prompt)
    rows.append((label, r.usage.input_tokens, r.usage.output_tokens, r.usage.cost))
    print(
        f"{label} | in={r.usage.input_tokens:>4} out={r.usage.output_tokens:>3} cost={r.usage.cost}"
    )

cheapest = min(rows, key=lambda row: row[3] or 0)
print()
print("cheapest per answer:", cheapest[0].strip())

**Notes**

- Token counts (and often prices) differ per model even for identical prompts —
  "which model" is a real engineering decision with data behind it.
- In Logfire's **LLMs view** this comparison accumulates automatically across runs:
  latency, tokens/sec, and cost per model over time.